In [1]:
import sys
import os
from matplotlib.colors import LinearSegmentedColormap
sys.path.append('/root/capsule/code/beh_ephys_analysis')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pynwb import NWBFile, TimeSeries, NWBHDF5IO
from scipy.io import loadmat
from scipy.stats import zscore
import ast
from pathlib import Path
import glob
import json
import seaborn as sns
from PyPDF2 import PdfMerger
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import re
from utils.beh_functions import session_dirs, parseSessionID, load_model_dv, makeSessionDF, get_session_tbl, get_unit_tbl, get_history_from_nwb
from utils.ephys_functions import*
from utils.ccf_utils import ccf_pts_convert_to_mm, pir_to_lps, project_to_plane
from utils.combine_tools import apply_qc, to_str_intlike
from utils.capsule_migration import capsule_directories
import pickle
import scipy.stats as stats
import spikeinterface as si
import shutil
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import r2_score
import warnings
from scipy.stats import gaussian_kde
import trimesh
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
from utils.ccf_utils import ccf_pts_convert_to_mm
from trimesh import load_mesh
from scipy.stats import pearsonr
import statsmodels.api as sm
from aind_ephys_utils import align
import k3d
warnings.filterwarnings('ignore')
capsule_dirs = capsule_directories()

%matplotlib inline

# Config

In [2]:
criteria_name = 'beh_all_DRN'
# Session-level behavior gate, matching F_ephys_behavior_action&outcome_DRN.ipynb.
# 'beh_none' is the ungated run (166 units / 38 sessions / 15 animals); 'beh_only' keeps only
# 23 units / 7 sessions / 4 animals, 17 of which come from 826159 alone -- too few to pick
# examples from, so the examples below are drawn from the beh_none feature table.
beh_criteria = 'beh_none'

combined_folder = os.path.join(capsule_dirs["manuscript_fig_dir"], f'F_ephys_beh_action&outcome_DRN_{beh_criteria}')
target_folder = os.path.join(combined_folder, 'example_figs')
if not os.path.exists(target_folder):
    os.makedirs(target_folder)

# Per-unit regression table written by F_ephys_behavior_action&outcome_DRN.ipynb. The examples
# below were picked from its T_response_svs_all / T_response_hit_all columns.
features_file = os.path.join(combined_folder, f'features_combined_{criteria_name}.pkl')
features = pd.read_pickle(features_file)
features['animal'] = features['session'].str.split('_').str[1]
print(f'{len(features)} units, {features.session.nunique()} sessions, {features.animal.nunique()} animals')
print(f'figures -> {target_folder}')

166 units, 38 sessions, 15 animals
figures -> /root/capsule/scratch/results/manuscript/figures/F_ephys_beh_action&outcome_DRN_beh_none/example_figs


## Candidate selection

`pick_examples` re-derives the example lists below from the combined feature table, so they can be
regenerated when QC changes. Ranks by |T| within a contrast, caps units per session so one session
cannot dominate, and optionally requires a minimum number of trials in the *minority* condition.

That last guard matters for this cohort: the DRN animals are perseverative (overall switch rate
~3-15%) and finish nearly every trial (finish rate 0.96-1.00), so the top-|T| unit often has only
~10 switch or ~3 miss trials and its raster is unreadable. `min_minority` is counted *after* the
drift cut. Leave `count_trials=False` for a fast ranking; counting loads every session table.

In [3]:
def pick_examples(mode, max_per_session=2, n=10, min_minority=15, count_trials=True, top_pool=45):
    """Rank units for the 'svs' or 'hit' contrast. Returns a DataFrame."""
    T, p = (f'T_response_{mode}_all', f'p_response_{mode}_all')
    cand = features[['session', 'animal', 'unit_id', T, p]].dropna(subset=[T]).copy()
    cand = cand[cand[p] < 0.05]
    cand['absT'] = cand[T].abs()
    cand = cand.sort_values('absT', ascending=False).head(top_pool)

    if count_trials:
        tcol = 'go_cue_time' if mode == 'svs' else 'goCue_start_time'
        tbl_cache = {}
        minority = []
        for _, r in cand.iterrows():
            sess, uid = r['session'], int(r['unit_id'])
            if sess not in tbl_cache:
                tbl_cache[sess] = makeSessionDF(sess) if mode == 'svs' else get_session_tbl(sess)
            sub = tbl_cache[sess]
            drift = load_drift(sess, uid, data_type='raw')
            if drift is not None:
                lo, hi = drift['ephys_cut']
                if lo is not None:
                    sub = sub[sub[tcol] >= lo]
                if hi is not None:
                    sub = sub[sub[tcol] <= hi]
            grp = np.asarray(sub['svs']).astype(bool) if mode == 'svs' else np.asarray(sub['animal_response'].values != 2)
            minority.append(min(int(grp.sum()), int((~grp).sum())))
        cand['n_minority'] = minority
        cand = cand[cand['n_minority'] >= min_minority]

    picked, per_session = [], {}
    for _, r in cand.iterrows():
        if per_session.get(r['session'], 0) >= max_per_session:
            continue
        per_session[r['session']] = per_session.get(r['session'], 0) + 1
        picked.append(r)
        if len(picked) >= n:
            break
    out = pd.DataFrame(picked)
    if len(out):
        print(f'{mode}: {len(out)} units, {out.session.nunique()} sessions, {out.animal.nunique()} animals')
        for _, r in out.iterrows():
            tail = f", n_minority={r['n_minority']}" if 'n_minority' in out.columns else ''
            print(f"            ('{r['session']}', {int(r['unit_id'])}),   # T={r[T]:+.2f}{tail}")
    return out

# candidates_svs = pick_examples('svs')
# candidates_hit = pick_examples('hit')

## Shared raster helper

Both contrasts use the same body, so it lives in one place here. Note `data_type='raw'`: none of the
38 DRN sessions have curated output, so the `'curated'` default used by the mPFC notebook would make
`get_unit_tbl` fail and -- worse -- make `load_drift` return `None` silently, skipping the drift cut.

In [4]:
def plot_example_raster(session, unit_id, mode, colormap, tb=-1, tf=0.5):
    """Raster + rate for one unit, split by stay/switch ('svs') or miss/hit ('hit')."""
    if mode == 'svs':
        session_df_curr = makeSessionDF(session)
        tcol, labels = 'go_cue_time', ['stay', 'switch']
    else:
        session_df_curr = get_session_tbl(session)
        tcol, labels = 'goCue_start_time', ['miss', 'hit']

    unit_tbl = get_unit_tbl(session, data_type='raw')
    match = unit_tbl['unit_id'] == unit_id
    if not match.any():
        print(f'  skipped: unit {unit_id} not in {session}')
        return None
    spike_times_curr = unit_tbl.loc[match, 'spike_times'].values[0].copy()

    # drift cut: ephys_cut is on the same (harp) clock as the trial times, not session-relative
    unit_drift = load_drift(session, unit_id, data_type='raw')
    if unit_drift is not None:
        if unit_drift['ephys_cut'][0] is not None:
            spike_times_curr = spike_times_curr[spike_times_curr >= unit_drift['ephys_cut'][0]]
            session_df_curr = session_df_curr[session_df_curr[tcol] >= unit_drift['ephys_cut'][0]]
        if unit_drift['ephys_cut'][1] is not None:
            spike_times_curr = spike_times_curr[spike_times_curr <= unit_drift['ephys_cut'][1]]
            session_df_curr = session_df_curr[session_df_curr[tcol] <= unit_drift['ephys_cut'][1]]

    if mode == 'svs':
        map_value = session_df_curr['svs']
    else:
        map_value = session_df_curr['animal_response'].values != 2
    n_minor = min(int(np.sum(np.asarray(map_value))), int(np.sum(~np.asarray(map_value).astype(bool))))

    fig = plt.figure(figsize=(4, 12))
    gs = gridspec.GridSpec(2, 1, height_ratios=[2, 1], hspace=0.4)
    fig, ax1, ax2 = plot_raster_rate(spike_times_curr, session_df_curr[tcol].values,
                                map_value, [-1, 0.5, 2], labels,
                                colormap, fig, gs[0], kernel=True, tb=tb, tf=tf,
                                tau_decay=0.08, tau_rise=0.001, step_size=0.01)
    ax2.set_ylabel('Firing rate (Hz)', fontsize=12)
    ax2.set_xlabel('Time from go cue (s)', fontsize=12)
    plt.suptitle(f'Session: {session}, Unit: {unit_id}', fontsize=14)
    plt.tight_layout()
    prefix = 'switch-stay' if mode == 'svs' else 'hit-miss'
    plt.savefig(os.path.join(target_folder, f'{prefix}_{session}_{unit_id}.pdf'))
    plt.savefig(os.path.join(target_folder, f'{prefix}_{session}_{unit_id}.svg'))
    print(f'  {session} u{unit_id}: {len(session_df_curr)} trials, {n_minor} in minority condition')
    return fig

# Stay vs Switch example

In [5]:
# From pick_examples('svs'): |T| ranked, <=2 per session, >=15 switch trials after the drift cut.
# 5 animals / 7 sessions. Sign of T is noted; negative = higher rate on stay trials.
examples = [('behavior_844917_2026-06-04_12-30-01', 55),    # T=+4.47, n_minority=15
            ('behavior_844917_2026-06-04_12-30-01', 206),   # T=+3.65, n_minority=15
            ('behavior_841598_2026-05-01_14-15-09', 76),    # T=+3.38, n_minority=46
            ('behavior_843661_2026-04-28_09-49-53', 194),   # T=+2.94, n_minority=35
            ('behavior_841598_2026-05-01_14-15-09', 91),    # T=+2.91, n_minority=46
            ('behavior_843661_2026-04-29_08-19-05', 172),   # T=-2.73, n_minority=62
            ('behavior_826159_2026-01-26_15-15-12', 62),    # T=+2.40, n_minority=19
            ('behavior_843661_2026-05-01_09-00-46', 105),   # T=+2.11, n_minority=48
            ('behavior_843661_2026-04-28_09-49-53', 231),   # T=+2.00, n_minority=35
            ('behavior_835444_2026-02-17_13-56-45', 141)]   # T=-1.97, n_minority=23

In [6]:
custom_cmap = LinearSegmentedColormap.from_list('switch', [(0.0, 'green'), (0.5, 'white'), (1.0, 'purple')])

In [7]:
for curr_ind, curr_example in enumerate(examples):
    session, unit_id = curr_example
    fig = plot_example_raster(session, unit_id, 'svs', custom_cmap)
    if fig is not None:
        plt.close(fig)

  behavior_844917_2026-06-04_12-30-01 u55: 437 trials, 15 in minority condition


  behavior_844917_2026-06-04_12-30-01 u206: 437 trials, 15 in minority condition


  behavior_841598_2026-05-01_14-15-09 u76: 342 trials, 46 in minority condition


  behavior_843661_2026-04-28_09-49-53 u194: 285 trials, 35 in minority condition


  behavior_841598_2026-05-01_14-15-09 u91: 342 trials, 46 in minority condition


  behavior_843661_2026-04-29_08-19-05 u172: 402 trials, 62 in minority condition


  behavior_826159_2026-01-26_15-15-12 u62: 333 trials, 19 in minority condition


  behavior_843661_2026-05-01_09-00-46 u105: 409 trials, 48 in minority condition


  behavior_843661_2026-04-28_09-49-53 u231: 285 trials, 35 in minority condition


  behavior_835444_2026-02-17_13-56-45 u141: 246 trials, 23 in minority condition


# Miss vs Hit

In [8]:
# From pick_examples('hit'): |T| ranked, <=2 per session, >=15 miss trials after the drift cut.
# 4 animals / 6 sessions. Misses are rare in this cohort (finish rate 0.96-1.00), which is why the
# threshold drops several larger-|T| units (e.g. 808650 u40, T=+3.71, had only 3 miss trials).
examples = [('behavior_838332_2026-03-12_13-31-08', 125),   # T=+6.97, n_minority=16
            ('behavior_838332_2026-03-13_12-29-22', 324),   # T=+6.38, n_minority=53
            ('behavior_838332_2026-03-11_13-01-00', 146),   # T=+5.95, n_minority=22
            ('behavior_835444_2026-02-20_12-12-35', 15),    # T=+4.24, n_minority=27
            ('behavior_838332_2026-03-12_13-31-08', 129),   # T=+3.95, n_minority=16
            ('behavior_838332_2026-03-11_13-01-00', 105),   # T=-3.79, n_minority=39
            ('behavior_835444_2026-02-20_12-12-35', 3),     # T=+3.61, n_minority=27
            ('behavior_841859_2026-04-09_09-36-29', 204),   # T=+3.11, n_minority=92
            ('behavior_841596_2026-03-25_14-10-54', 3),     # T=+2.73, n_minority=33
            ('behavior_838332_2026-03-13_12-29-22', 94)]    # T=-2.37, n_minority=67

In [9]:
custom_cmap = LinearSegmentedColormap.from_list('hit', [(0.0, 'blue'), (0.5, 'white'), (1.0, 'orange')])

In [10]:
for curr_ind, curr_example in enumerate(examples):
    session, unit_id = curr_example
    fig = plot_example_raster(session, unit_id, 'hit', custom_cmap)
    if fig is not None:
        plt.close(fig)

  behavior_838332_2026-03-12_13-31-08 u125: 400 trials, 16 in minority condition


  behavior_838332_2026-03-13_12-29-22 u324: 249 trials, 53 in minority condition


  behavior_838332_2026-03-11_13-01-00 u146: 340 trials, 22 in minority condition


  behavior_835444_2026-02-20_12-12-35 u15: 431 trials, 27 in minority condition


  behavior_838332_2026-03-12_13-31-08 u129: 400 trials, 16 in minority condition


  behavior_838332_2026-03-11_13-01-00 u105: 443 trials, 39 in minority condition


  behavior_835444_2026-02-20_12-12-35 u3: 431 trials, 27 in minority condition


  behavior_841859_2026-04-09_09-36-29 u204: 404 trials, 92 in minority condition


  behavior_841596_2026-03-25_14-10-54 u3: 360 trials, 33 in minority condition


  behavior_838332_2026-03-13_12-29-22 u94: 370 trials, 67 in minority condition
